# Challenge 6: Production-Ready Dashboard

You've been asked to take an existing dashboard from development to production. This means:
- Writing optimized datasets for dashboard widgets
- Understanding credential modes and sharing
- Setting up monitoring with audit logs
- Preparing a deployment configuration (Declarative Automation Bundle)

You'll practice:
- Writing dashboard-quality SQL datasets (aggregations, time-series, Top-N)
- Choosing the correct credential mode for different scenarios
- Querying `system.access.audit` for dashboard monitoring
- Completing a Declarative Automation Bundle YAML configuration

> **Instructions:** Run the setup cell below, then complete each task. Replace `<FILL_IN>` with the correct SQL or values. Tasks 1-4 are SQL (run in the SQL Editor). Tasks 5-7 are conceptual (fill in answers inline).

---

In [0]:
%run ./challenge_6_setup

## Task 1: Write a KPI Counter Dataset

Write a SQL query that returns a **single row** with these metrics for a Counter widget:
- `total_revenue` — sum of net_revenue
- `total_orders` — count of orders
- `avg_order_value` — average revenue per order
- `profit_margin_pct` — `(SUM(net_revenue) - SUM(cost)) / SUM(net_revenue) * 100`

Filter to **2024 Q4 only** (October-December).

---

In [0]:
%sql
-- Task 1: KPI Counter dataset for Q4 2024

SELECT
  ROUND(<FILL_IN>, 0) AS total_revenue,
  <FILL_IN> AS total_orders,
  ROUND(<FILL_IN>, 2) AS avg_order_value,
  ROUND(<FILL_IN>, 1) AS profit_margin_pct
FROM gold_sales
WHERE order_date >= '<FILL_IN>'
  AND order_date < '<FILL_IN>';

## Task 2: Write a Time-Series Dataset (Line Chart)

Write a SQL query for a **monthly revenue trend** grouped by channel. This will power a multi-line chart showing how each channel performs over time.

Return: `order_month`, `channel`, `total_revenue`, `order_count`

Sort by month ascending.

---

In [0]:
%sql
-- Task 2: Monthly revenue trend by channel (for a multi-line chart)

SELECT
  <FILL_IN> AS order_month,
  <FILL_IN>,
  ROUND(SUM(<FILL_IN>), 2) AS total_revenue,
  COUNT(<FILL_IN>) AS order_count
FROM gold_sales
GROUP BY <FILL_IN>, <FILL_IN>
ORDER BY <FILL_IN> ASC;

## Task 3: Write a Top-N Dataset (Ranked Table)

Write a SQL query showing the **Top 5 customers by lifetime revenue** from the `gold_customers` table.

Return: `customer_name`, `segment`, `region`, `lifetime_orders`, `lifetime_revenue`

Follow dashboard best practices: select only needed columns, use aliases, include LIMIT.

---

In [0]:
%sql
-- Task 3: Top 5 customers by lifetime revenue (for a ranked table widget)

SELECT
  <FILL_IN>,
  <FILL_IN>,
  <FILL_IN>,
  <FILL_IN>,
  ROUND(<FILL_IN>, 2) AS lifetime_revenue
FROM <FILL_IN>
ORDER BY <FILL_IN>
LIMIT <FILL_IN>;

## Task 4: Write an Audit Monitoring Query

Write a query against `system.access.audit` to find the **top 10 most active dashboard viewers** in the last 30 days.

Return: `viewer` (email), `dashboards_viewed` (count of distinct dashboards), `total_views` (total view events).

Filter to dashboard view actions only.

> **Note:** This requires system table access. If unavailable, write the query anyway for practice.

---

In [0]:
%sql
-- Task 4: Top 10 most active dashboard viewers (last 30 days)
-- Requires system.access.audit access

SELECT
  <FILL_IN> AS viewer,
  COUNT(DISTINCT <FILL_IN>) AS dashboards_viewed,
  COUNT(*) AS total_views
FROM <FILL_IN>
WHERE action_name IN (<FILL_IN>)
  AND event_date >= CURRENT_DATE() - INTERVAL <FILL_IN>
GROUP BY <FILL_IN>
ORDER BY total_views DESC
LIMIT 10;

## Task 5: Choose the Correct Credential Mode

For each scenario below, fill in whether you'd use **"Shared"** or **"Individual"** data permissions:

| Scenario | Credential Mode |
| --- | --- |
| Executive KPI dashboard — all execs see same numbers | `<FILL_IN>` |
| Regional sales dashboard — each manager sees only their region (UC permissions control access) | `<FILL_IN>` |
| Dashboard embedded in company intranet portal | `<FILL_IN>` |
| Compliance dashboard — viewers must only see data they're authorized for | `<FILL_IN>` |
| Development/testing during dashboard build | `<FILL_IN>` |

---

## Task 6: Design a Scheduling Strategy

You're deploying the sales dashboard to production. Design three schedules:

| Schedule Name | Frequency | Filters | Subscribers | Purpose |
| --- | --- | --- | --- | --- |
| `<FILL_IN>` | `<FILL_IN>` | None (all data) | `<FILL_IN>` | Morning briefing for leadership |
| `<FILL_IN>` | `<FILL_IN>` | Region = "Northeast" | `<FILL_IN>` | Weekly report for NE team |
| `<FILL_IN>` | `<FILL_IN>` | None | `<FILL_IN>` | Keep cache warm for fast loads |

---

## Task 7: Complete the Bundle Configuration

Fill in the missing values in this Declarative Automation Bundle YAML to deploy the sales dashboard:

```yaml
bundle:
  name: <FILL_IN>

resources:
  dashboards:
    sales_dashboard:
      display_name: "<FILL_IN>"
      file_path: <FILL_IN>
      warehouse_id: ${var.sql_warehouse_id}
      permissions:
        - level: <FILL_IN>
          group_name: executives
        - level: <FILL_IN>
          group_name: analytics_team

targets:
  dev:
    default: true
    workspace:
      host: https://dev-workspace.cloud.databricks.com

  prod:
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    variables:
      sql_warehouse_id: "<FILL_IN>"
```

Fill in:
1. Bundle name (snake_case project identifier)
2. Dashboard display name
3. File path to the `.lvdash.json` file
4. Permission level for executives (view only)
5. Permission level for analytics team (full edit)
6. Production warehouse ID placeholder

---

---

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# STOP HERE — Solutions Below

Only scroll down after you've attempted all 7 tasks!

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

---

# Solutions

---

In [0]:
%sql
-- Solution: Task 1 - KPI Counter dataset for Q4 2024
SELECT
  ROUND(SUM(net_revenue), 0) AS total_revenue,
  COUNT(order_id) AS total_orders,
  ROUND(SUM(net_revenue) / COUNT(order_id), 2) AS avg_order_value,
  ROUND((SUM(net_revenue) - SUM(cost)) / SUM(net_revenue) * 100, 1) AS profit_margin_pct
FROM gold_sales
WHERE order_date >= '2024-10-01'
  AND order_date < '2025-01-01';

In [0]:
%sql
-- Solution: Task 2 - Monthly revenue trend by channel
SELECT
  DATE_TRUNC('month', order_date) AS order_month,
  channel,
  ROUND(SUM(net_revenue), 2) AS total_revenue,
  COUNT(order_id) AS order_count
FROM gold_sales
GROUP BY DATE_TRUNC('month', order_date), channel
ORDER BY order_month ASC;

In [0]:
%sql
-- Solution: Task 3 - Top 5 customers by lifetime revenue
SELECT
  customer_name,
  segment,
  region,
  lifetime_orders,
  ROUND(lifetime_revenue, 2) AS lifetime_revenue
FROM gold_customers
ORDER BY lifetime_revenue DESC
LIMIT 5;

In [0]:
%sql
-- Solution: Task 4 - Top 10 most active dashboard viewers
SELECT
  user_identity.email AS viewer,
  COUNT(DISTINCT request_params.dashboard_id) AS dashboards_viewed,
  COUNT(*) AS total_views
FROM system.access.audit
WHERE action_name IN ('getDashboard', 'getPublishedDashboard')
  AND event_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY user_identity.email
ORDER BY total_views DESC
LIMIT 10;

### Solution: Task 5 - Credential Modes

| Scenario | Credential Mode |
| --- | --- |
| Executive KPI dashboard — all execs see same numbers | **Shared** |
| Regional sales dashboard — each manager sees only their region | **Individual** |
| Dashboard embedded in company intranet portal | **Shared** (with service principal) |
| Compliance dashboard — viewers must only see authorized data | **Individual** |
| Development/testing during dashboard build | **Shared** |

### Solution: Task 6 - Scheduling Strategy

| Schedule Name | Frequency | Filters | Subscribers | Purpose |
| --- | --- | --- | --- | --- |
| **Executive Daily** | Daily at 7:00 AM | None (all data) | CEO, CFO, VP Sales (email) | Morning briefing for leadership |
| **NE Weekly Report** | Weekly (Monday 9 AM) | Region = "Northeast" | NE Sales Manager (email) | Weekly report for NE team |
| **Cache Warming** | Every 2 hours | None | No subscribers | Keep cache warm for fast loads |

### Solution: Task 7 - Bundle Configuration

```yaml
bundle:
  name: sales-dashboard-project

resources:
  dashboards:
    sales_dashboard:
      display_name: "Sales Performance Dashboard"
      file_path: ./dashboards/sales_report.lvdash.json
      warehouse_id: ${var.sql_warehouse_id}
      permissions:
        - level: CAN_VIEW
          group_name: executives
        - level: CAN_EDIT
          group_name: analytics_team

targets:
  dev:
    default: true
    workspace:
      host: https://dev-workspace.cloud.databricks.com

  prod:
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    variables:
      sql_warehouse_id: "your-prod-warehouse-id"
```

**Key answers:**
1. Bundle name: `sales-dashboard-project` (snake_case)
2. Display name: Any descriptive name ("Sales Performance Dashboard")
3. File path: `./dashboards/sales_report.lvdash.json` (relative to bundle root)
4. Executives: `CAN_VIEW` (view only)
5. Analytics team: `CAN_EDIT` (full edit access)
6. Warehouse ID: The actual warehouse ID from your prod environment